# Swin temporal stable OOF diagnostics

Creates validation OOF diagnostics from the 5 fold Swin temporal stable checkpoints. No retraining.


In [ ]:
from pathlib import Path
import subprocess
import sys
import shutil


def install_p100_torch_if_needed():
    try:
        gpu = subprocess.check_output(
            ['bash', '-lc', "nvidia-smi --query-gpu=name --format=csv,noheader | head -1"],
            text=True,
        ).strip()
    except Exception:
        gpu = ''
    if 'P100' in gpu:
        print(f'Installing a Pascal-compatible PyTorch build for {gpu}')
        subprocess.run(
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '-q',
                '--force-reinstall',
                'torch==2.5.1',
                'torchvision==0.20.1',
                '--index-url',
                'https://download.pytorch.org/whl/cu121',
            ],
            check=True,
        )
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'pillow==11.3.0'],
            check=True,
        )
    return gpu


gpu_name = install_p100_torch_if_needed()

REPO = Path('/kaggle/working/solafune-nowcast')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/shionsuio/solafune-nowcast.git', str(REPO)], check=True)

sys.path.insert(0, str(REPO / 'src'))

import torch
print('GPU:', gpu_name or (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'), 'torch:', torch.__version__)

from run_oof_diagnostics import run

DATA_ROOT = Path('/kaggle/input/solafune-dataset-v2')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('/kaggle/input/datasets/suioshion/solafune-dataset-v2')

MODEL_ROOT = Path('/tmp/combined_swin_temporal_stable_models')
MODEL_ROOT.mkdir(exist_ok=True)

def copy_first(pattern, destination_name, preferred_parts):
    candidates = sorted(Path('/kaggle/input').rglob(pattern))
    def score(path):
        text = str(path)
        for index, part in enumerate(preferred_parts):
            if part in text:
                return index
        return 999
    candidates = sorted(candidates, key=score)
    print(destination_name, candidates[:5])
    assert candidates, f'{pattern} not found'
    target = MODEL_ROOT / destination_name
    target.write_bytes(candidates[0].read_bytes())

for fold in [0, 1]:
    copy_first(f'best_fold{fold}.pth', f'best_fold{fold}.pth', ['solafune-swin-temporal-full'])
    copy_first(f'band_stats_fold{fold}.json', f'band_stats_fold{fold}.json', ['solafune-swin-temporal-full'])
copy_first('best_fold2.pth', 'best_fold2.pth', ['swin-temporal-stable-fold2'])
copy_first('band_stats_fold2.json', 'band_stats_fold2.json', ['swin-temporal-stable-fold2'])
for fold in [3, 4]:
    copy_first(f'best_fold{fold}.pth', f'best_fold{fold}.pth', ['swin-temporal-models-fold34'])
    copy_first(f'band_stats_fold{fold}.json', f'band_stats_fold{fold}.json', ['swin-temporal-models-fold34'])

print('DATA_ROOT:', DATA_ROOT)
print('MODEL_ROOT:', MODEL_ROOT, sorted(p.name for p in MODEL_ROOT.iterdir()))


class Args:
    root = '/kaggle/working'
    kaggle_input_root = str(DATA_ROOT)
    checkpoint_root = str(MODEL_ROOT)
    folds = '0,1,2,3,4'
    batch_size = 16
    workers = 2
    model_subdir = 'swin_v2_temporal_stable'
    output_dir = 'outputs/oof_swin_v2_temporal_stable'
    use_temporal_differences = True
    use_temporal_summary = True
    use_location_features = False
    location_metadata_path = None
    no_amp = True


output_dir = run(Args())

import pandas as pd
for name in ['oof_overall_summary.csv', 'oof_fold_summary.csv', 'oof_location_summary.csv', 'oof_month_summary.csv', 'oof_satellite_month_summary.csv']:
    print('\n', name)
    display(pd.read_csv(Path(output_dir) / name).head(30))

# Do not save large checkpoints as Kaggle output artifacts.
shutil.rmtree('/kaggle/working/models', ignore_errors=True)
shutil.rmtree(MODEL_ROOT, ignore_errors=True)
